# 아산시 돌봄 데이터 병합 파이프라인

`preprocessing.ipynb` 실행 후 생성된 전처리 결과물을 읽어서 3개 마스터 테이블을 생성합니다.

| 마스터 | PK | 조인 대상 | 분석 용도 |
|---|---|---|---|
| A. 읍면동 마스터 | 읍면동 × 연도 | 인구 + 고령화율 + 시설 | 읍면동 간 비교, 돌봄 사각지대 |
| B. 아산시 시계열 | 연도 | 요양등급 + 수급자 + 건강조사 | 연도별 추이, 수요 증가 분석 |
| C. 충남 전망 | 연도 | 장래인구추계 | 2052년까지 수요 예측 |

## 0. 환경 설정 & 전처리 결과 로딩

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)

DATA_DIR = Path(r'C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\데이터')
PRE_DIR = DATA_DIR.parent / 'preprocessed'   # preprocessing.ipynb 산출물
OUT_DIR = DATA_DIR.parent / 'master'
OUT_DIR.mkdir(exist_ok=True)

print(f'전처리 결과: {PRE_DIR}')
print(f'마스터 출력: {OUT_DIR}')

전처리 결과: C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\preprocessed
마스터 출력: C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\master


In [14]:
# ── 전처리 결과 일괄 로딩 ──
df_census       = pd.read_csv(PRE_DIR / '01_인구총조사_long.csv', encoding="cp949")
df_resident     = pd.read_csv(PRE_DIR / '02_주민등록인구_long.csv', encoding="cp949")
df_proj         = pd.read_csv(PRE_DIR / '03_장래인구추계_long.csv', encoding="cp949")
df_ltc          = pd.read_csv(PRE_DIR / '04_장기요양_long.csv', encoding="cp949")
df_facility     = pd.read_csv(PRE_DIR / '05_재가노인복지시설_아산.csv', encoding="cp949")
df_pop_unified  = pd.read_csv(PRE_DIR / '06_읍면동별_인구통합.csv', encoding="cp949")
df_aging        = pd.read_csv(PRE_DIR / '07_읍면동별_고령화율.csv', encoding="cp949")
df_ltc_summary  = pd.read_csv(PRE_DIR / '08_장기요양_아산_등급추이.csv', encoding="cp949")
df_proj_aging   = pd.read_csv(PRE_DIR / '09_충남_장래_고령화율.csv', encoding="cp949")
df_access       = pd.read_csv(PRE_DIR / '10_시설접근성.csv', encoding="cp949")

# 건강조사/수급자는 있으면 로딩
health_path = PRE_DIR / '11_지역사회건강조사_아산.csv'
df_health = df_health = pd.read_csv(health_path, encoding="cp949") if health_path.exists() else None

welfare_files = sorted((PRE_DIR).glob('*수급자*'))
df_welfare = pd.read_csv(welfare_files[0], encoding='cp949') if welfare_files else None
print('로딩 완료:')
for name, df in [('인구총조사', df_census), ('주민등록', df_resident), ('장래인구', df_proj),
                  ('장기요양', df_ltc), ('시설', df_facility), ('건강조사', df_health), ('수급자', df_welfare)]:
    print(f'  {name}: {df.shape if df is not None else "미로딩"}')

로딩 완료:
  인구총조사: (680, 4)
  주민등록: (14382, 5)
  장래인구: (2691, 5)
  장기요양: (1253, 7)
  시설: (84, 6)
  건강조사: (21492, 36)
  수급자: (13641, 7)


---
## 1. 마스터 A — 읍면동 마스터

**PK:** `읍면동 × 연도`

| 조인 소스 | 키 | 가져오는 컬럼 |
|---|---|---|
| 주민등록인구 | 읍면동, 연도 | 총인구, 유소년, 생산가능 |
| 고령화율 | 읍면동, 연도 | 고령인구, 고령화율 |
| 재가시설 | 읍면동 (cross) | 유형별 시설수 |

In [15]:
# ── 베이스: 주민등록 총인구 ──
base = df_resident[
    (df_resident['연령별'] == '합계') & (df_resident['항목'] == '총인구(명)')
][['읍면동', '연도', '값']].rename(columns={'값': '총인구'})

# ── 고령인구 조인 ──
master_a = base.merge(
    df_aging[['읍면동', '연도', '고령인구', '고령화율']],
    on=['읍면동', '연도'], how='left'
)

# ── 유소년/생산가능인구 조인 ──
for label, col_name in [('15세미만', '유소년인구'), ('15~64세', '생산가능인구')]:
    tmp = df_resident[
        (df_resident['연령별'] == label) & (df_resident['항목'] == '총인구(명)')
    ][['읍면동', '연도', '값']].rename(columns={'값': col_name})
    master_a = master_a.merge(tmp, on=['읍면동', '연도'], how='left')

# ── 파생 지표 ──
master_a['고령화지수'] = (master_a['고령인구'] / master_a['유소년인구'] * 100).round(1)
master_a['노년부양비'] = (master_a['고령인구'] / master_a['생산가능인구'] * 100).round(1)

print(f'인구 베이스: {master_a.shape}')
master_a.head()

인구 베이스: (68, 9)


,읍면동,연도,총인구,고령인구,고령화율,유소년인구,생산가능인구,고령화지수,노년부양비
0,염치읍,2015,7643.0,1581.0,20.69,1032.0,5030.0,153.2,31.4
1,배방읍,2015,68783.0,4158.0,6.05,15159.0,49466.0,27.4,8.4
2,송악면,2015,4140.0,1145.0,27.66,439.0,2556.0,260.8,44.8
3,탕정면,2015,31421.0,1202.0,3.83,5326.0,24893.0,22.6,4.8
4,음봉면,2015,17827.0,1808.0,10.14,2609.0,13410.0,69.3,13.5


In [16]:
# ── 시설 데이터 조인 (최신 연도에만 부착) ──
fac_count = df_facility.groupby(['읍면동', '종류']).size().unstack(fill_value=0)
fac_count.columns = [f'시설_{c}' for c in fac_count.columns]
fac_count['시설_합계'] = fac_count.sum(axis=1)
fac_count = fac_count.reset_index()

latest_year = master_a['연도'].max()
master_a = master_a.merge(fac_count, on='읍면동', how='left')

# 시설 데이터는 2024 기준 스냅샷 → 과거 연도 행에는 NaN
fac_cols = [c for c in master_a.columns if c.startswith('시설_')]
master_a.loc[master_a['연도'] != latest_year, fac_cols] = np.nan
master_a['인구천명당_시설수'] = (master_a['시설_합계'] / master_a['총인구'] * 1000).round(2)

master_a = master_a.sort_values(['읍면동', '연도']).reset_index(drop=True)

print(f'마스터 A 완성: {master_a.shape}')
print(f'읍면동 {master_a["읍면동"].nunique()}개 × 연도 {sorted(master_a["연도"].unique())}')

마스터 A 완성: (68, 16)
읍면동 17개 × 연도 [np.int64(2015), np.int64(2020), np.int64(2023), np.int64(2024)]


In [17]:
# ── 마스터 A 검증: 최신 연도 고령화율 Top 5 ──
latest = master_a[master_a['연도'] == master_a['연도'].max()]
print(f'=== {latest["연도"].iloc[0]}년 고령화율 Top 5 ===')
latest.nlargest(5, '고령화율')[
    ['읍면동', '총인구', '고령인구', '고령화율', '고령화지수', '노년부양비', '시설_합계', '인구천명당_시설수']
]

=== 2024년 고령화율 Top 5 ===


,읍면동,총인구,고령인구,고령화율,고령화지수,노년부양비,시설_합계,인구천명당_시설수
3,도고면,4533.0,2018.0,44.52,1391.7,85.1,6.0,1.32
15,선장면,3159.0,1375.0,43.53,2148.4,79.9,NaN,NaN
19,송악면,3860.0,1464.0,37.93,590.3,68.2,1.0,0.26
31,영인면,6084.0,2157.0,35.45,941.9,58.3,1.0,0.16
27,염치읍,5851.0,1871.0,31.98,540.8,51.5,2.0,0.34


---
## 2. 마스터 B — 아산시 시계열

**PK:** `연도`

| 조인 소스 | 집계 방법 | 가져오는 컬럼 |
|---|---|---|
| 마스터 A | groupby 연도 sum | 총인구, 고령인구 등 |
| 장기요양 등급 | pivot (등급→컬럼) | 등급별 인원 |
| 건강조사 | groupby year agg | 응답자수, 평균연령 등 |
| 기초생활 수급자 | groupby 연도 mean | 연평균 수급자수 |

In [18]:
# ── 1) 마스터 A에서 아산시 전체 인구 집계 ──
pop_by_year = master_a.groupby('연도').agg(
    총인구=('총인구', 'sum'),
    고령인구=('고령인구', 'sum'),
    유소년인구=('유소년인구', 'sum'),
    생산가능인구=('생산가능인구', 'sum'),
).reset_index()
pop_by_year['고령화율'] = (pop_by_year['고령인구'] / pop_by_year['총인구'] * 100).round(2)

print(f'인구 집계: {pop_by_year.shape}')
pop_by_year

인구 집계: (4, 6)


,연도,총인구,고령인구,유소년인구,생산가능인구,고령화율
0,2015,319929.0,33269.0,53976.0,232684.0,10.40
1,2020,340518.0,41700.0,52554.0,246264.0,12.25
2,2023,376632.0,49666.0,53252.0,273714.0,13.19
3,2024,392509.0,53048.0,53482.0,285979.0,13.52


In [19]:
# ── 2) 장기요양 등급 pivot ──
ltc_total = df_ltc_summary[df_ltc_summary['성별'] == '합계'].pivot_table(
    index='연도', columns='등급', values='인원', aggfunc='sum'
).reset_index()
ltc_total.columns = ['연도'] + [f'요양_{c}' for c in ltc_total.columns[1:]]

print(f'장기요양 피벗: {ltc_total.shape}')
ltc_total.tail()

장기요양 피벗: (11, 9)


,연도,요양_1등급,요양_2등급,요양_3등급,요양_4등급,요양_5등급,요양_계,요양_등급외,요양_인지지원등급
6,2020,179.0,495.0,1192.0,1668.0,470.0,4701.0,555.0,142.0
7,2021,222.0,515.0,1321.0,1894.0,582.0,5306.0,586.0,186.0
8,2022,213.0,513.0,1314.0,2021.0,685.0,5589.0,617.0,226.0
9,2023,258.0,518.0,1437.0,2111.0,820.0,6070.0,632.0,294.0
10,2024,264.0,505.0,1543.0,2418.0,939.0,6546.0,603.0,274.0


In [20]:
# ── 3) 인구 + 장기요양 병합 ──
master_b = pop_by_year.merge(ltc_total, on='연도', how='outer')
master_b = master_b.sort_values('연도').reset_index(drop=True)

print(f'인구 + 요양: {master_b.shape}')
master_b

인구 + 요양: (11, 14)


,연도,총인구,고령인구,유소년인구,생산가능인구,고령화율,요양_1등급,요양_2등급,요양_3등급,요양_4등급,요양_5등급,요양_계,요양_등급외,요양_인지지원등급
0,2014,NaN,NaN,NaN,NaN,NaN,135.0,387.0,689.0,510.0,53.0,2424.0,650.0,NaN
1,2015,319929.0,33269.0,53976.0,232684.0,10.40,149.0,404.0,730.0,624.0,93.0,2638.0,638.0,NaN
2,2016,NaN,NaN,NaN,NaN,NaN,165.0,413.0,783.0,707.0,148.0,2857.0,641.0,NaN
3,2017,NaN,NaN,NaN,NaN,NaN,183.0,436.0,783.0,976.0,253.0,3326.0,695.0,NaN
4,2018,NaN,NaN,NaN,NaN,NaN,193.0,463.0,841.0,1164.0,322.0,3687.0,646.0,58.0
5,2019,NaN,NaN,NaN,NaN,NaN,189.0,471.0,1051.0,1438.0,402.0,4229.0,583.0,95.0
6,2020,340518.0,41700.0,52554.0,246264.0,12.25,179.0,495.0,1192.0,1668.0,470.0,4701.0,555.0,142.0
7,2021,NaN,NaN,NaN,NaN,NaN,222.0,515.0,1321.0,1894.0,582.0,5306.0,586.0,186.0
8,2022,NaN,NaN,NaN,NaN,NaN,213.0,513.0,1314.0,2021.0,685.0,5589.0,617.0,226.0
9,2023,376632.0,49666.0,53252.0,273714.0,13.19,258.0,518.0,1437.0,2111.0,820.0,6070.0,632.0,294.0


In [21]:
# ── 4) 건강조사 집계 조인 (있으면) ──
if df_health is not None:
    # 컬럼명 확인 (한글 매핑 여부에 따라 다름)
    print(f'건강조사 컬럼: {list(df_health.columns[:10])}')

    year_col = 'year' if 'year' in df_health.columns else '조사연도'
    age_col = 'age' if 'age' in df_health.columns else '나이'

    health_agg = df_health.groupby(year_col).agg(
        건강조사_응답자수=(year_col, 'size'),
        평균연령=(age_col, 'mean'),
    ).round(1).reset_index().rename(columns={year_col: '연도'})
    master_b = master_b.merge(health_agg, on='연도', how='left')
    print(f'건강조사 조인 완료: {health_agg.shape[0]}개 연도')
else:
    print('⚠ 건강조사 미로딩')

건강조사 컬럼: ['나이', '당뇨병 현재 치료', '당뇨 합병증 경험', '저혈당 경험', '평생 음주 경험', '1회 음주량', '교육수준', '고혈압 현재 치료', '인플루엔자 접종', '인플루엔자 접종 비용']
건강조사 조인 완료: 12개 연도


In [22]:
# ── 5) 수급자 집계 조인 (있으면) ──
if df_welfare is not None:
    welfare_agg = df_welfare.groupby('연도').agg(
        수급자수_연평균=('수급자수', 'mean'),
    ).round(0).reset_index()
    master_b = master_b.merge(welfare_agg, on='연도', how='left')
    print(f'수급자 조인 완료: {welfare_agg.shape[0]}개 연도')
else:
    print('⚠ 수급자 미로딩 — preprocessing에서 수급자 CSV 생성 필요')

print(f'\n최종 마스터 B: {master_b.shape}')
master_b

수급자 조인 완료: 11개 연도

최종 마스터 B: (11, 17)


,연도,총인구,고령인구,유소년인구,생산가능인구,고령화율,요양_1등급,요양_2등급,요양_3등급,요양_4등급,요양_5등급,요양_계,요양_등급외,요양_인지지원등급,건강조사_응답자수,평균연령,수급자수_연평균
0,2014,NaN,NaN,NaN,NaN,NaN,135.0,387.0,689.0,510.0,53.0,2424.0,650.0,NaN,1793,48.9,59.0
1,2015,319929.0,33269.0,53976.0,232684.0,10.40,149.0,404.0,730.0,624.0,93.0,2638.0,638.0,NaN,1793,49.3,62.0
2,2016,NaN,NaN,NaN,NaN,NaN,165.0,413.0,783.0,707.0,148.0,2857.0,641.0,NaN,1791,51.2,75.0
3,2017,NaN,NaN,NaN,NaN,NaN,183.0,436.0,783.0,976.0,253.0,3326.0,695.0,NaN,1792,51.2,72.0
4,2018,NaN,NaN,NaN,NaN,NaN,193.0,463.0,841.0,1164.0,322.0,3687.0,646.0,58.0,1793,52.1,72.0
5,2019,NaN,NaN,NaN,NaN,NaN,189.0,471.0,1051.0,1438.0,402.0,4229.0,583.0,95.0,1793,53.3,85.0
6,2020,340518.0,41700.0,52554.0,246264.0,12.25,179.0,495.0,1192.0,1668.0,470.0,4701.0,555.0,142.0,1784,53.3,94.0
7,2021,NaN,NaN,NaN,NaN,NaN,222.0,515.0,1321.0,1894.0,582.0,5306.0,586.0,186.0,1798,52.8,105.0
8,2022,NaN,NaN,NaN,NaN,NaN,213.0,513.0,1314.0,2021.0,685.0,5589.0,617.0,226.0,1791,54.1,115.0
9,2023,376632.0,49666.0,53252.0,273714.0,13.19,258.0,518.0,1437.0,2111.0,820.0,6070.0,632.0,294.0,1785,54.3,124.0


---
## 3. 마스터 C — 충남 전망

**PK:** `연도`

장래인구추계에서 총인구/고령/유소년/생산가능 인구를 산출하고 고령화 지표를 파생합니다.

⚠ **80세이상 소계 제외 필수** — 개별 5세 단위(80-84, 85-89...)와 중복됨

In [23]:
# ── 총인구 ──
proj_total = df_proj[
    (df_proj['연령대'] == '계') & (df_proj['성별'] == '계')
][['연도', '인구수']].rename(columns={'인구수': '총인구_추계'})

# ── 고령인구 (65+ 개별, 80세이상 소계 제외) ──
elderly_labels = [a for a in df_proj['연령대'].unique() if any(
    a.startswith(f'{x} ') for x in range(65, 100)
)] + ['100세 이상']
elderly_labels = [a for a in elderly_labels if a != '80세이상']

proj_elderly = df_proj[
    (df_proj['연령대'].isin(elderly_labels)) & (df_proj['성별'] == '계')
].groupby('연도')['인구수'].sum().reset_index().rename(columns={'인구수': '고령인구_추계'})

# ── 유소년 (0-14) ──
youth_labels = [a for a in df_proj['연령대'].unique() if any(
    a.startswith(f'{x} ') for x in range(0, 15)
)]
proj_youth = df_proj[
    (df_proj['연령대'].isin(youth_labels)) & (df_proj['성별'] == '계')
].groupby('연도')['인구수'].sum().reset_index().rename(columns={'인구수': '유소년인구_추계'})

# ── 생산가능 (15-64) ──
working_labels = [a for a in df_proj['연령대'].unique() if any(
    a.startswith(f'{x} ') for x in range(15, 65)
)]
proj_working = df_proj[
    (df_proj['연령대'].isin(working_labels)) & (df_proj['성별'] == '계')
].groupby('연도')['인구수'].sum().reset_index().rename(columns={'인구수': '생산가능인구_추계'})

print(f'고령 연령대: {elderly_labels}')
print(f'유소년 연령대: {youth_labels}')

고령 연령대: ['65 - 69세', '70 - 74세', '75 - 79세', '80 - 84세', '85 - 89세', '90 - 94세', '95 - 99세', '100세 이상']
유소년 연령대: ['0 - 4세', '5 - 9세', '10 - 14세']


In [24]:
# ── 병합 & 파생 지표 ──
master_c = (proj_total
    .merge(proj_elderly, on='연도')
    .merge(proj_youth, on='연도')
    .merge(proj_working, on='연도')
)

master_c['고령화율_추계'] = (master_c['고령인구_추계'] / master_c['총인구_추계'] * 100).round(2)
master_c['고령화지수_추계'] = (master_c['고령인구_추계'] / master_c['유소년인구_추계'] * 100).round(1)
master_c['노년부양비_추계'] = (master_c['고령인구_추계'] / master_c['생산가능인구_추계'] * 100).round(1)

print(f'마스터 C: {master_c.shape} | {master_c["연도"].min()}~{master_c["연도"].max()}')

마스터 C: (39, 8) | 2014~2052


In [25]:
# ── 마스터 C 검증: 주요 시점 ──
master_c[master_c['연도'].isin([2024, 2026, 2030, 2035, 2040, 2045, 2050])]

,연도,총인구_추계,고령인구_추계,유소년인구_추계,생산가능인구_추계,고령화율_추계,고령화지수_추계,노년부양비_추계
10,2024,2223801,460769,243591,1519441,20.72,189.2,30.3
12,2026,2233847,514938,222641,1496268,23.05,231.3,34.4
16,2030,2253655,602247,186463,1464945,26.72,323.0,41.1
21,2035,2271887,710994,168446,1392447,31.30,422.1,51.1
26,2040,2274956,819216,174483,1281257,36.01,469.5,63.9
31,2045,2256841,893141,176206,1187494,39.57,506.9,75.2
36,2050,2211343,948457,166381,1096505,42.89,570.1,86.5


---
## 4. 저장

In [27]:
master_a.to_csv(OUT_DIR / 'master_a_읍면동.csv', index=False, encoding='cp949')
master_b.to_csv(OUT_DIR / 'master_b_아산시_시계열.csv', index=False, encoding='cp949')
master_c.to_csv(OUT_DIR / 'master_c_충남_전망.csv', index=False, encoding='cp949')

print(f'✓ master_a_읍면동.csv          ({master_a.shape[0]}행 × {master_a.shape[1]}열)')
print(f'✓ master_b_아산시_시계열.csv    ({master_b.shape[0]}행 × {master_b.shape[1]}열)')
print(f'✓ master_c_충남_전망.csv        ({master_c.shape[0]}행 × {master_c.shape[1]}열)')
print(f'\n저장 경로: {OUT_DIR}')

✓ master_a_읍면동.csv          (68행 × 16열)
✓ master_b_아산시_시계열.csv    (11행 × 17열)
✓ master_c_충남_전망.csv        (39행 × 8열)

저장 경로: C:\Users\HP\IdeaProjects\sundo\asan_care\asan_care\master
